In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import torch
import cvxpy as cp
from cvxpylayers.torch import CvxpyLayer
import yfinance as yf
import random
from dataclasses import dataclass
from typing import Dict, Tuple
from tqdm import tqdm
import itertools

### Data Collection

In [2]:

symbols = [
    "CL=F","NG=F","RB=F","HO=F",
    "GC=F","SI=F","HG=F","PL=F","PA=F",
    "ZC=F","ZS=F","ZW=F",
    "KC=F","SB=F","CT=F","CC=F","OJ=F",
    "LE=F","HE=F"
]

start_date = "1999-12-01"
end_date   = "2023-12-31"


data = yf.download(symbols, start=start_date, end=end_date)
price_df = data["Close"]

price_df = price_df.dropna(axis=1, thresh=int(0.95 * len(price_df)))
price_df = price_df.ffill()
price_df = price_df.dropna()

print("Final universe size:", price_df.shape[1])
ret_df = price_df / price_df.shift(1).dropna()

ret_df = ret_df[10:]
ret_df

[*********************100%***********************]  19 of 19 completed

Final universe size: 15


Ticker,CC=F,CL=F,CT=F,GC=F,HE=F,HG=F,HO=F,KC=F,NG=F,RB=F,SB=F,SI=F,ZC=F,ZS=F,ZW=F
Date,,,,,,,,,,,,,,,
2001-01-02,1.026385,1.014925,0.969011,0.986765,1.002058,0.966332,0.954997,0.967201,0.855652,1.011708,0.997059,0.991276,1.000000,1.002004,1.016499
2001-01-03,0.967866,1.027574,1.010108,0.998510,1.000000,1.006724,0.993070,1.000789,0.982783,1.018868,1.005900,0.987899,1.000000,0.987500,0.994590
2001-01-04,1.019920,1.008945,0.977854,0.997388,1.006160,0.989678,1.001628,0.998424,1.094890,1.010617,1.011730,1.009354,1.000000,1.011646,1.026292
2001-01-05,1.058594,0.992908,0.996813,1.002619,1.004082,1.020859,1.001510,1.015785,1.027778,1.002321,0.992271,1.005075,1.000000,0.988488,1.004417
2001-01-08,1.000000,0.976786,1.013632,1.000000,0.997967,1.001202,0.957681,0.992230,1.048649,1.017672,1.019474,0.997805,1.000000,0.988354,0.989446
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2023-12-22,1.005364,0.995534,1.007962,1.008827,1.009908,0.996674,0.986799,0.995868,1.014774,0.986843,1.018775,0.999301,1.001058,1.001927,1.006122
2023-12-26,0.993505,1.027325,1.003636,1.000535,0.971268,1.000000,1.002856,1.008039,0.977012,1.013239,0.995635,0.993989,1.015328,1.010387,1.032454
2023-12-27,0.999300,0.980680,1.005996,1.011515,1.008297,1.012322,0.983176,1.017494,1.027059,0.998471,1.003410,1.009568,0.992192,1.002665,0.979175


In [3]:
TRADING_DAYS = 252
WINDOW_TREND = 252
LAMBDA = 0.94
EPS = 1e-6
init_w = 252

DELTA = 50.0          # 논문 δ=50
REFIT_YEARS = 2       # 2년마다 refit
INIT_TRAIN_YEARS = 14 # 논문은 14년 initial train (우리도 2000~ 기준으로 맞춤)

TORCH_DTYPE = torch.double
DEVICE = "cpu"

In [4]:
assets = ret_df.columns
dates = ret_df.index
N = len(assets)
T = len(dates)

trend_df = ret_df.rolling(WINDOW_TREND).mean().shift(1)

R = ret_df.to_numpy(dtype=float) 

V_list, L_list, date_list = [], [], []


# 초기 공분산: 초반 252일 표본공분산으로 시작 (안정적)

V = np.cov(R[:init_w].T, bias=False)  # (N,N)

for t in range(init_w, T):
    r_prev = R[t-1:t, :].T  # (N,1)
    V = LAMBDA * V + (1 - LAMBDA) * (r_prev @ r_prev.T)

    Vj = V + EPS * np.eye(N)

    try:
        L_low = np.linalg.cholesky(Vj)
        L = L_low.T
    except np.linalg.LinAlgError:
        w, Q = np.linalg.eigh(V)
        w = np.clip(w, EPS, None)
        Vj = (Q * w) @ Q.T
        L_low = np.linalg.cholesky(Vj)
        L = L_low.T


    V_list.append(Vj)
    L_list.append(L)
    date_list.append(dates[t])

Vhat = np.stack(V_list, axis=0)   
Lhat = np.stack(L_list, axis=0)  
cov_idx = pd.Index(date_list, name="Date")

In [5]:
# EMA 공분산이 존재하는 날짜만 사용
trend_df2 = trend_df.loc[cov_idx].dropna()

# trend_df2의 인덱스에 맞춰 Lhat도 슬라이싱
# (cov_idx와 trend_df2.index가 부분집합 관계일 수 있음)
mask = cov_idx.isin(trend_df2.index)
Lhat2 = Lhat[mask]
Vhat2 = Vhat[mask]
dates2 = cov_idx[mask]

# 예측/목표 수익률도 같은 날짜로 맞춤
ret_df2 = ret_df.loc[dates2]
X = trend_df2.to_numpy()
Y = ret_df2.to_numpy()


### Experiment 1 : Unconstrained + Univariate

In [6]:
def build_mvo_layer_unconstrained(n_assets: int, delta: float):
    z = cp.Variable(n_assets)
    mu = cp.Parameter(n_assets)              # y_hat_t
    L  = cp.Parameter((n_assets, n_assets))  # cov sqrt

    obj = cp.Minimize(-mu @ z + 0.5 * delta * cp.sum_squares(L @ z))
    prob = cp.Problem(obj, [])
    return CvxpyLayer(prob, parameters=[mu, L], variables=[z])



def predict_mu(X: torch.Tensor, theta: torch.Tensor) -> torch.Tensor:
    # X: (T, N), theta: (N,)
    return X * theta  # elementwise



def fit_theta_ols_through_origin(X_np: np.ndarray, Y_np: np.ndarray, ridge: float = 1e-8) -> np.ndarray:
    # X_np, Y_np: (T, N)
    # theta_j = (x^T y) / (x^T x)
    num = (X_np * Y_np).sum(axis=0)
    den = (X_np * X_np).sum(axis=0) + ridge
    theta = num / den
    return theta



def fit_theta_ipo(
    layer,
    X_train_np: np.ndarray,         # (Ttr, N)
    Y_train_np: np.ndarray,         # (Ttr, N)  (next-day realized)
    L_train_np: np.ndarray,         # (Ttr, N, N)
    delta: float,
    lr: float = 0.01,
    epochs: int = 100,
    batch_size: int = 256,
    seed: int = 0,
    block_id: int | None = None,
    n_blocks: int | None = None,
) -> np.ndarray:
    torch.manual_seed(seed)
    np.random.seed(seed)

    X = torch.tensor(X_train_np, dtype=TORCH_DTYPE, device=DEVICE)
    Y = torch.tensor(Y_train_np, dtype=TORCH_DTYPE, device=DEVICE)
    L = torch.tensor(L_train_np, dtype=TORCH_DTYPE, device=DEVICE)

    Ttr, N = X.shape
    theta = torch.zeros(N, dtype=TORCH_DTYPE, device=DEVICE, requires_grad=True)

    opt = torch.optim.Adam([theta], lr=lr)

    # mini-batch indices
    idx = np.arange(Ttr)

    desc = f"IPO lr={lr}, batch={batch_size}"
    if (block_id is not None) and (n_blocks is not None):
        desc += f" [Block {block_id}/{n_blocks}]"

    for ep in tqdm(range(epochs), desc = desc, leave =True):
        np.random.shuffle(idx)
        for start in range(0, Ttr, batch_size):
            b = idx[start:start+batch_size]
            Xb = X[b]
            Yb = Y[b]
            Lb = L[b]

            mu_hat = predict_mu(Xb, theta)              # (B, N)
            (z_star,) = layer(mu_hat, Lb)               # (B, N)

            # cost = -y^T z + (delta/2)||L z||^2
            ret_term = -(Yb * z_star).sum(dim=1)        # (B,)
            risk_term = 0.5 * delta * (Lb @ z_star.unsqueeze(-1)).squeeze(-1).pow(2).sum(dim=1)  # (B,)
            loss = (ret_term + risk_term).mean()

            opt.zero_grad()
            loss.backward()
            opt.step()

    return theta.detach().cpu().numpy()




def make_refit_blocks(dates: pd.Index, init_train_years: int, refit_years: int):
    # dates are daily; we block by calendar years
    years = pd.Index(dates.year)
    start_year = years.min()
    oos_start_year = start_year + init_train_years

    blocks = []
    cur = oos_start_year
    last_year = years.max()

    while cur <= last_year:
        block_start = pd.Timestamp(f"{cur}-01-01")
        block_end   = pd.Timestamp(f"{cur + refit_years}-01-01")  # exclusive
        blocks.append((block_start, block_end))
        cur += refit_years

    return blocks




def backtest_exp_i_univariate_unconstrained(
    trend_df2: pd.DataFrame,
    ret_df: pd.DataFrame,
    Lhat2: np.ndarray,
    dates2: pd.Index,
    delta: float = DELTA,
    lr: float = 0.01,
    epochs: int = 120,
    batch_size: int = 256,
    seed: int = 0
):
    ret_next = ret_df.shift(-1).reindex(dates2)

    valid = ret_next.notna().all(axis=1)
    dates_bt = dates2[valid.values]

    X_df = trend_df2.reindex(dates_bt)
    Y_df = ret_next.reindex(dates_bt)

    mask = pd.Index(dates2).isin(dates_bt)
    L_bt = Lhat2[mask]

    X_np = X_df.to_numpy(dtype=float)
    Y_np = Y_df.to_numpy(dtype=float)

    N = X_np.shape[1]
    layer = build_mvo_layer_unconstrained(N, delta)

    blocks = make_refit_blocks(dates_bt, INIT_TRAIN_YEARS, REFIT_YEARS)
    
    trainable_blocks = []
    for (b_start, b_end) in blocks:
        in_block = (dates_bt >= b_start) & (dates_bt < b_end)
        train_mask = (dates_bt < b_start)

        if in_block.sum() == 0:
            continue
        if train_mask.sum() < 500:
            continue

        trainable_blocks.append((b_start, b_end))

    n_blocks = len(trainable_blocks)

    port_ols = pd.Series(index=dates_bt, dtype=float)
    port_ipo = pd.Series(index=dates_bt, dtype=float)

    X_torch = torch.tensor(X_np, dtype=TORCH_DTYPE)
    Y_torch = torch.tensor(Y_np, dtype=TORCH_DTYPE)
    L_torch = torch.tensor(L_bt, dtype=TORCH_DTYPE)

    for bi, (b_start, b_end) in enumerate(trainable_blocks, start=1):
        in_block = (dates_bt >= b_start) & (dates_bt < b_end)
        if in_block.sum() == 0:
            continue

        train_mask = (dates_bt < b_start)
        if train_mask.sum() < 500:
            continue

        X_tr = X_np[train_mask]
        Y_tr = Y_np[train_mask]
        L_tr = L_bt[train_mask]

        theta_ols = fit_theta_ols_through_origin(X_tr, Y_tr)
        theta_ipo = fit_theta_ipo(
            layer, X_tr, Y_tr, L_tr, delta,
            lr=lr, epochs=epochs, batch_size=batch_size, seed=seed, block_id = bi, n_blocks = n_blocks
        )

        b_idx = np.where(in_block)[0]

        with torch.no_grad():
            theta_ols_t = torch.tensor(theta_ols, dtype=TORCH_DTYPE)
            theta_ipo_t = torch.tensor(theta_ipo, dtype=TORCH_DTYPE)

            mu_ols = X_torch[b_idx] * theta_ols_t
            mu_ipo = X_torch[b_idx] * theta_ipo_t

            (z_ols,) = layer(mu_ols, L_torch[b_idx])
            (z_ipo,) = layer(mu_ipo, L_torch[b_idx])

            r_ols = (z_ols * Y_torch[b_idx]).sum(dim=1).cpu().numpy()
            r_ipo = (z_ipo * Y_torch[b_idx]).sum(dim=1).cpu().numpy()

        port_ols.iloc[b_idx] = r_ols
        port_ipo.iloc[b_idx] = r_ipo

    return port_ols.dropna(), port_ipo.dropna()




def compute_table5_metrics(port_ret: pd.Series, delta: float = 50.0, var_q: float = 0.05) -> dict:
    s = port_ret.dropna()
    r = s.to_numpy(dtype=float)

    mu_d = r.mean()
    sig_d = r.std(ddof=1)

    annual_return = mu_d * TRADING_DAYS
    annual_vol = sig_d * np.sqrt(TRADING_DAYS)
    sharpe = np.nan if annual_vol == 0 else annual_return / annual_vol

    # drawdown series
    equity = (1.0 + s).cumprod()
    peak = equity.cummax()
    dd = equity / peak - 1.0
    avg_drawdown = dd.mean()

    # VaR (daily)
    value_at_risk = np.quantile(r, var_q)

    # MVO cost (daily-based, paper eq.36 form)
    mvo_cost = -mu_d + 0.5 * delta * (sig_d ** 2)

    return {
        "Annual return": annual_return,
        "Sharpe ratio": sharpe,
        "Volatility": annual_vol,
        "Avg drawdown": avg_drawdown,
        "Value at risk": value_at_risk,
        "MVO cost": mvo_cost,
    }


def make_table5(port_ipo: pd.Series, port_ols: pd.Series, delta: float = 50.0, var_q: float = 0.05) -> pd.DataFrame:
    m_ipo = compute_table5_metrics(port_ipo, delta=delta, var_q=var_q)
    m_ols = compute_table5_metrics(port_ols, delta=delta, var_q=var_q)

    table = pd.DataFrame([m_ipo, m_ols], index=["IPO", "OLS"])
    return table


In [7]:
grid = [
    {"lr":0.001, "batch":128},
    {"lr":0.001,  "batch":256},
    {"lr":0.005,  "batch":128},
    {"lr":0.005,  "batch":256},
    {"lr":0.01,  "batch":128},
    {"lr":0.01,  "batch":256},
]

rows = []
for g in grid:
    port_ols, port_ipo = backtest_exp_i_univariate_unconstrained(
        trend_df2=trend_df2, ret_df=ret_df, Lhat2=Lhat2, dates2=dates2,
        delta=DELTA, lr=g["lr"], epochs=120, batch_size=g["batch"], seed=0
    )
    table = make_table5(port_ipo=port_ipo, port_ols=port_ols, delta=DELTA, var_q=0.05)

    rows.append({
        "lr": g["lr"],
        "batch": g["batch"],
        "IPO Sharpe": table.loc["IPO","Sharpe ratio"],
        "IPO AnnRet": table.loc["IPO","Annual return"],
        "IPO Vol": table.loc["IPO","Volatility"],
        "OLS Sharpe": table.loc["OLS","Sharpe ratio"],
        "OLS AnnRet": table.loc["OLS","Annual return"],
        "OLS Vol": table.loc["OLS","Volatility"],
    })

result_df = pd.DataFrame(rows).sort_values(["IPO Sharpe"], ascending=False)
display(result_df)


/opt/anaconda3/envs/ml_env/lib/python3.11/site-packages/cvxpylayers/torch/cvxpylayer.py:429: UserWarning: Sparse CSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/SparseCsrTensorImpl.cpp:51.)
  torch_csr = torch.sparse_csr_tensor(
IPO lr=0.001, batch=128:  54%|█████▍    | 65/120 [03:30<02:55,  3.19s/it] 

### Experiment 3 : Equality Constraints + Univariate

In [ ]:
def build_mvo_layer_equality(n_assets: int, delta: float):
    z = cp.Variable(n_assets)
    mu = cp.Parameter(n_assets)
    L  = cp.Parameter((n_assets, n_assets))  # upper chol

    objective = cp.Minimize(
        -mu @ z + 0.5 * delta * cp.sum_squares(L @ z)
    )

    constraints = [
        cp.sum(z) == 0
    ]

    prob = cp.Problem(objective, constraints)
    return CvxpyLayer(prob, parameters=[mu, L], variables=[z])




def predict_mu(X: torch.Tensor, theta: torch.Tensor) -> torch.Tensor:
    # X: (T, N), theta: (N,)
    return X * theta  # elementwise



def fit_theta_ols_through_origin(X_np: np.ndarray, Y_np: np.ndarray, ridge: float = 1e-8) -> np.ndarray:
    # X_np, Y_np: (T, N)
    # theta_j = (x^T y) / (x^T x)
    num = (X_np * Y_np).sum(axis=0)
    den = (X_np * X_np).sum(axis=0) + ridge
    theta = num / den
    return theta



def fit_theta_ipo(
    layer,
    X_train_np: np.ndarray,         # (Ttr, N)
    Y_train_np: np.ndarray,         # (Ttr, N)  (next-day realized)
    L_train_np: np.ndarray,         # (Ttr, N, N)
    delta: float,
    lr: float = 0.01,
    epochs: int = 100,
    batch_size: int = 256,
    seed: int = 0,
    block_id: int | None = None,
    n_blocks: int | None = None,
) -> np.ndarray:
    torch.manual_seed(seed)
    np.random.seed(seed)

    X = torch.tensor(X_train_np, dtype=TORCH_DTYPE, device=DEVICE)
    Y = torch.tensor(Y_train_np, dtype=TORCH_DTYPE, device=DEVICE)
    L = torch.tensor(L_train_np, dtype=TORCH_DTYPE, device=DEVICE)

    Ttr, N = X.shape
    theta = torch.zeros(N, dtype=TORCH_DTYPE, device=DEVICE, requires_grad=True)

    opt = torch.optim.Adam([theta], lr=lr)

    # mini-batch indices
    idx = np.arange(Ttr)

    desc = f"IPO lr={lr}, batch={batch_size}"
    if (block_id is not None) and (n_blocks is not None):
        desc += f" [Block {block_id}/{n_blocks}]"    

    for ep in tqdm(range(epochs), desc = desc, leave = True):
        np.random.shuffle(idx)
        for start in range(0, Ttr, batch_size):
            b = idx[start:start+batch_size]
            Xb = X[b]
            Yb = Y[b]
            Lb = L[b]

            mu_hat = predict_mu(Xb, theta)              # (B, N)
            (z_star,) = layer(mu_hat, Lb)               # (B, N)

            # cost = -y^T z + (delta/2)||L z||^2
            ret_term = -(Yb * z_star).sum(dim=1)        # (B,)
            risk_term = 0.5 * delta * (Lb @ z_star.unsqueeze(-1)).squeeze(-1).pow(2).sum(dim=1)  # (B,)
            loss = (ret_term + risk_term).mean()

            opt.zero_grad()
            loss.backward()
            opt.step()

    return theta.detach().cpu().numpy()




def make_refit_blocks(dates: pd.Index, init_train_years: int, refit_years: int):
    # dates are daily; we block by calendar years
    years = pd.Index(dates.year)
    start_year = years.min()
    oos_start_year = start_year + init_train_years

    blocks = []
    cur = oos_start_year
    last_year = years.max()

    while cur <= last_year:
        block_start = pd.Timestamp(f"{cur}-01-01")
        block_end   = pd.Timestamp(f"{cur + refit_years}-01-01")  # exclusive
        blocks.append((block_start, block_end))
        cur += refit_years

    return blocks




def backtest_exp_i_univariate_equality(
    trend_df2: pd.DataFrame,
    ret_df: pd.DataFrame,
    Lhat2: np.ndarray,
    dates2: pd.Index,
    delta: float = DELTA,
    lr: float = 0.01,
    epochs: int = 120,
    batch_size: int = 256,
    seed: int = 0
):
    ret_next = ret_df.shift(-1).reindex(dates2)

    valid = ret_next.notna().all(axis=1)
    dates_bt = dates2[valid.values]

    X_df = trend_df2.reindex(dates_bt)
    Y_df = ret_next.reindex(dates_bt)

    mask = pd.Index(dates2).isin(dates_bt)
    L_bt = Lhat2[mask]

    X_np = X_df.to_numpy(dtype=float)
    Y_np = Y_df.to_numpy(dtype=float)

    N = X_np.shape[1]
    layer = build_mvo_layer_equality(N, delta)

    blocks = make_refit_blocks(dates_bt, INIT_TRAIN_YEARS, REFIT_YEARS)

    trainable_blocks = []
    for (b_start, b_end) in blocks:
        in_block = (dates_bt >= b_start) & (dates_bt < b_end)
        train_mask = (dates_bt < b_start)

        if in_block.sum() == 0:
            continue
        if train_mask.sum() < 500:
            continue

        trainable_blocks.append((b_start, b_end))

    n_blocks = len(trainable_blocks)    

    port_ols = pd.Series(index=dates_bt, dtype=float)
    port_ipo = pd.Series(index=dates_bt, dtype=float)

    X_torch = torch.tensor(X_np, dtype=TORCH_DTYPE)
    Y_torch = torch.tensor(Y_np, dtype=TORCH_DTYPE)
    L_torch = torch.tensor(L_bt, dtype=TORCH_DTYPE)

    for bi, (b_start, b_end) in enumerate(trainable_blocks, start=1):
        in_block = (dates_bt >= b_start) & (dates_bt < b_end)
        if in_block.sum() == 0:
            continue

        train_mask = (dates_bt < b_start)
        if train_mask.sum() < 500:
            continue

        X_tr = X_np[train_mask]
        Y_tr = Y_np[train_mask]
        L_tr = L_bt[train_mask]

        theta_ols = fit_theta_ols_through_origin(X_tr, Y_tr)
        theta_ipo = fit_theta_ipo(
            layer, X_tr, Y_tr, L_tr, delta,
            lr=lr, epochs=epochs, batch_size=batch_size, seed=seed, block_id = bi, n_blocks = n_blocks
        )
        
        b_idx = np.where(in_block)[0]

        with torch.no_grad():
            theta_ols_t = torch.tensor(theta_ols, dtype=TORCH_DTYPE)
            theta_ipo_t = torch.tensor(theta_ipo, dtype=TORCH_DTYPE)

            mu_ols = X_torch[b_idx] * theta_ols_t
            mu_ipo = X_torch[b_idx] * theta_ipo_t

            (z_ols,) = layer(mu_ols, L_torch[b_idx])
            (z_ipo,) = layer(mu_ipo, L_torch[b_idx])

            r_ols = (z_ols * Y_torch[b_idx]).sum(dim=1).cpu().numpy()
            r_ipo = (z_ipo * Y_torch[b_idx]).sum(dim=1).cpu().numpy()

        port_ols.iloc[b_idx] = r_ols
        port_ipo.iloc[b_idx] = r_ipo

    return port_ols.dropna(), port_ipo.dropna()




def compute_table5_metrics(port_ret: pd.Series, delta: float = 50.0, var_q: float = 0.05) -> dict:
    s = port_ret.dropna()
    r = s.to_numpy(dtype=float)

    mu_d = r.mean()
    sig_d = r.std(ddof=1)

    annual_return = mu_d * TRADING_DAYS
    annual_vol = sig_d * np.sqrt(TRADING_DAYS)
    sharpe = np.nan if annual_vol == 0 else annual_return / annual_vol

    # drawdown series
    equity = (1.0 + s).cumprod()
    peak = equity.cummax()
    dd = equity / peak - 1.0
    avg_drawdown = dd.mean()

    # VaR (daily)
    value_at_risk = np.quantile(r, var_q)

    # MVO cost (daily-based, paper eq.36 form)
    mvo_cost = -mu_d + 0.5 * delta * (sig_d ** 2)

    return {
        "Annual return": annual_return,
        "Sharpe ratio": sharpe,
        "Volatility": annual_vol,
        "Avg drawdown": avg_drawdown,
        "Value at risk": value_at_risk,
        "MVO cost": mvo_cost,
    }


def make_table5(port_ipo: pd.Series, port_ols: pd.Series, delta: float = 50.0, var_q: float = 0.05) -> pd.DataFrame:
    m_ipo = compute_table5_metrics(port_ipo, delta=delta, var_q=var_q)
    m_ols = compute_table5_metrics(port_ols, delta=delta, var_q=var_q)

    table = pd.DataFrame([m_ipo, m_ols], index=["IPO", "OLS"])
    return table


In [ ]:
grid = [
    {"lr":0.001, "batch":128},
    {"lr":0.001,  "batch":256},
    {"lr":0.005,  "batch":128},
    {"lr":0.005,  "batch":256},
    {"lr":0.01,  "batch":128},
    {"lr":0.01,  "batch":256},
]

rows = []
for g in grid:
    port_ols, port_ipo = backtest_exp_i_univariate_equality(
        trend_df2=trend_df2, ret_df=ret_df, Lhat2=Lhat2, dates2=dates2,
        delta=DELTA, lr=g["lr"], epochs=120, batch_size=g["batch"], seed=0
    )
    table = make_table5(port_ipo=port_ipo, port_ols=port_ols, delta=DELTA, var_q=0.05)

    rows.append({
        "lr": g["lr"],
        "batch": g["batch"],
        "IPO Sharpe": table.loc["IPO","Sharpe ratio"],
        "IPO AnnRet": table.loc["IPO","Annual return"],
        "IPO Vol": table.loc["IPO","Volatility"],
        "OLS Sharpe": table.loc["OLS","Sharpe ratio"],
        "OLS AnnRet": table.loc["OLS","Annual return"],
        "OLS Vol": table.loc["OLS","Volatility"],
    })

result_df = pd.DataFrame(rows).sort_values(["IPO Sharpe"], ascending=False)
display(result_df)


OLS daily mean/std: -0.0003897383481925284 0.018349989521941834
IPO daily mean/std: -3.456267094494936e-05 0.049246332968093255
     Annual return  Sharpe ratio  Volatility  Avg drawdown  Value at risk  \
IPO        -0.0087       -0.0111      0.7818       -0.6500        -0.0788   
OLS        -0.0982       -0.3372      0.2913       -0.3752        -0.0133   

     MVO cost  
IPO    0.0607  
OLS    0.0088  


### Experiment 5 : Inequality Constraints

In [ ]:
def build_mvo_layer_inequality(n_assets: int, delta: float, box: float):
    z = cp.Variable(n_assets)
    mu = cp.Parameter(n_assets)
    L  = cp.Parameter((n_assets, n_assets))  # upper chol

    objective = cp.Minimize(
        -mu @ z + 0.5 * delta * cp.sum_squares(L @ z)
    )

    constraints = [
        cp.sum(z) == 0,
        z <= box,
        z >= -box
    ]

    prob = cp.Problem(objective, constraints)
    return CvxpyLayer(prob, parameters=[mu, L], variables=[z])





def predict_mu(X: torch.Tensor, theta: torch.Tensor) -> torch.Tensor:
    # X: (T, N), theta: (N,)
    return X * theta  # elementwise



def fit_theta_ols_through_origin(X_np: np.ndarray, Y_np: np.ndarray, ridge: float = 1e-8) -> np.ndarray:
    # X_np, Y_np: (T, N)
    # theta_j = (x^T y) / (x^T x)
    num = (X_np * Y_np).sum(axis=0)
    den = (X_np * X_np).sum(axis=0) + ridge
    theta = num / den
    return theta



def fit_theta_ipo(
    layer,
    X_train_np: np.ndarray,         # (Ttr, N)
    Y_train_np: np.ndarray,         # (Ttr, N)  (next-day realized)
    L_train_np: np.ndarray,         # (Ttr, N, N)
    delta: float,
    lr: float = 0.01,
    epochs: int = 100,
    batch_size: int = 256,
    seed: int = 0,
    block_id: int | None = None,
    n_blocks: int | None = None,    
) -> np.ndarray:
    torch.manual_seed(seed)
    np.random.seed(seed)

    X = torch.tensor(X_train_np, dtype=TORCH_DTYPE, device=DEVICE)
    Y = torch.tensor(Y_train_np, dtype=TORCH_DTYPE, device=DEVICE)
    L = torch.tensor(L_train_np, dtype=TORCH_DTYPE, device=DEVICE)

    Ttr, N = X.shape
    theta = torch.zeros(N, dtype=TORCH_DTYPE, device=DEVICE, requires_grad=True)

    opt = torch.optim.Adam([theta], lr=lr)

    # mini-batch indices
    idx = np.arange(Ttr)

    desc = f"IPO lr={lr}, batch={batch_size}"
    if (block_id is not None) and (n_blocks is not None):
        desc += f" [Block {block_id}/{n_blocks}]"   

    for ep in tqdm(range(epochs), desc = desc, leave = True):
        np.random.shuffle(idx)
        for start in range(0, Ttr, batch_size):
            b = idx[start:start+batch_size]
            Xb = X[b]
            Yb = Y[b]
            Lb = L[b]

            mu_hat = predict_mu(Xb, theta)              # (B, N)
            (z_star,) = layer(mu_hat, Lb)               # (B, N)

            # cost = -y^T z + (delta/2)||L z||^2
            ret_term = -(Yb * z_star).sum(dim=1)        # (B,)
            risk_term = 0.5 * delta * (Lb @ z_star.unsqueeze(-1)).squeeze(-1).pow(2).sum(dim=1)  # (B,)
            loss = (ret_term + risk_term).mean()

            opt.zero_grad()
            loss.backward()
            opt.step()

    return theta.detach().cpu().numpy()




def make_refit_blocks(dates: pd.Index, init_train_years: int, refit_years: int):
    # dates are daily; we block by calendar years
    years = pd.Index(dates.year)
    start_year = years.min()
    oos_start_year = start_year + init_train_years

    blocks = []
    cur = oos_start_year
    last_year = years.max()

    while cur <= last_year:
        block_start = pd.Timestamp(f"{cur}-01-01")
        block_end   = pd.Timestamp(f"{cur + refit_years}-01-01")  # exclusive
        blocks.append((block_start, block_end))
        cur += refit_years

    return blocks




def backtest_exp_v_univariate_inequality(
    trend_df2: pd.DataFrame,
    ret_df: pd.DataFrame,
    Lhat2: np.ndarray,
    dates2: pd.Index,
    delta: float = DELTA,
    box: float = 0.1,
    lr: float = 0.01,
    epochs: int = 120,
    batch_size: int = 256,
    seed: int = 0
):
    ret_next = ret_df.shift(-1).reindex(dates2)

    valid = ret_next.notna().all(axis=1)
    dates_bt = dates2[valid.values]

    X_df = trend_df2.reindex(dates_bt)
    Y_df = ret_next.reindex(dates_bt)

    mask = pd.Index(dates2).isin(dates_bt)
    L_bt = Lhat2[mask]

    X_np = X_df.to_numpy(dtype=float)
    Y_np = Y_df.to_numpy(dtype=float)

    N = X_np.shape[1]
    layer = build_mvo_layer_inequality(N, delta, box)

    blocks = make_refit_blocks(dates_bt, INIT_TRAIN_YEARS, REFIT_YEARS)

    trainable_blocks = []
    for (b_start, b_end) in blocks:
        in_block = (dates_bt >= b_start) & (dates_bt < b_end)
        train_mask = (dates_bt < b_start)

        if in_block.sum() == 0:
            continue
        if train_mask.sum() < 500:
            continue

        trainable_blocks.append((b_start, b_end))

    n_blocks = len(trainable_blocks)
    
    port_ols = pd.Series(index=dates_bt, dtype=float)
    port_ipo = pd.Series(index=dates_bt, dtype=float)

    X_torch = torch.tensor(X_np, dtype=TORCH_DTYPE)
    Y_torch = torch.tensor(Y_np, dtype=TORCH_DTYPE)
    L_torch = torch.tensor(L_bt, dtype=TORCH_DTYPE)

    for bi, (b_start, b_end) in enumerate(trainable_blocks, start=1):
        in_block = (dates_bt >= b_start) & (dates_bt < b_end)
        if in_block.sum() == 0:
            continue

        train_mask = (dates_bt < b_start)
        if train_mask.sum() < 500:
            continue

        X_tr = X_np[train_mask]
        Y_tr = Y_np[train_mask]
        L_tr = L_bt[train_mask]

        theta_ols = fit_theta_ols_through_origin(X_tr, Y_tr)
        theta_ipo = fit_theta_ipo(
            layer, X_tr, Y_tr, L_tr, delta,
            lr=lr, epochs=epochs, batch_size=batch_size, seed=seed, block_id = bi, n_blocks = n_blocks
        )

        b_idx = np.where(in_block)[0]

        with torch.no_grad():
            theta_ols_t = torch.tensor(theta_ols, dtype=TORCH_DTYPE)
            theta_ipo_t = torch.tensor(theta_ipo, dtype=TORCH_DTYPE)

            mu_ols = X_torch[b_idx] * theta_ols_t
            mu_ipo = X_torch[b_idx] * theta_ipo_t

            (z_ols,) = layer(mu_ols, L_torch[b_idx])
            (z_ipo,) = layer(mu_ipo, L_torch[b_idx])

            r_ols = (z_ols * Y_torch[b_idx]).sum(dim=1).cpu().numpy()
            r_ipo = (z_ipo * Y_torch[b_idx]).sum(dim=1).cpu().numpy()

        port_ols.iloc[b_idx] = r_ols
        port_ipo.iloc[b_idx] = r_ipo

    return port_ols.dropna(), port_ipo.dropna()




def compute_table5_metrics(port_ret: pd.Series, delta: float = 50.0, var_q: float = 0.05) -> dict:
    s = port_ret.dropna()
    r = s.to_numpy(dtype=float)

    mu_d = r.mean()
    sig_d = r.std(ddof=1)

    annual_return = mu_d * TRADING_DAYS
    annual_vol = sig_d * np.sqrt(TRADING_DAYS)
    sharpe = np.nan if annual_vol == 0 else annual_return / annual_vol

    # drawdown series
    equity = (1.0 + s).cumprod()
    peak = equity.cummax()
    dd = equity / peak - 1.0
    avg_drawdown = dd.mean()

    # VaR (daily)
    value_at_risk = np.quantile(r, var_q)

    # MVO cost (daily-based, paper eq.36 form)
    mvo_cost = -mu_d + 0.5 * delta * (sig_d ** 2)

    return {
        "Annual return": annual_return,
        "Sharpe ratio": sharpe,
        "Volatility": annual_vol,
        "Avg drawdown": avg_drawdown,
        "Value at risk": value_at_risk,
        "MVO cost": mvo_cost,
    }


def make_table5(port_ipo: pd.Series, port_ols: pd.Series, delta: float = 50.0, var_q: float = 0.05) -> pd.DataFrame:
    m_ipo = compute_table5_metrics(port_ipo, delta=delta, var_q=var_q)
    m_ols = compute_table5_metrics(port_ols, delta=delta, var_q=var_q)

    table = pd.DataFrame([m_ipo, m_ols], index=["IPO", "OLS"])
    return table


In [ ]:
grid = [
    {"lr":0.001, "batch":128},
    {"lr":0.001,  "batch":256},
    {"lr":0.005,  "batch":128},
    {"lr":0.005,  "batch":256},
    {"lr":0.01,  "batch":128},
    {"lr":0.01,  "batch":256},
]

rows = []
for g in grid:
    port_ols, port_ipo = backtest_exp_v_univariate_inequality(
        trend_df2=trend_df2, ret_df=ret_df, Lhat2=Lhat2, dates2=dates2,
        delta=DELTA, box=0.125,
        lr=g["lr"], epochs=120, batch_size=g["batch"], seed=0
    )
    table = make_table5(port_ipo=port_ipo, port_ols=port_ols, delta=DELTA, var_q=0.05)

    rows.append({
        "lr": g["lr"],
        "batch": g["batch"],
        "IPO Sharpe": table.loc["IPO","Sharpe ratio"],
        "IPO AnnRet": table.loc["IPO","Annual return"],
        "IPO Vol": table.loc["IPO","Volatility"],
        "OLS Sharpe": table.loc["OLS","Sharpe ratio"],
        "OLS AnnRet": table.loc["OLS","Annual return"],
        "OLS Vol": table.loc["OLS","Volatility"],
    })

result_df = pd.DataFrame(rows).sort_values(["IPO Sharpe"], ascending=False)
display(result_df)


OLS daily mean/std: -0.0002750381452054639 0.004991169076293144
IPO daily mean/std: -0.00030677905676535264 0.012513473438094853
     Annual return  Sharpe ratio  Volatility  Avg drawdown  Value at risk  \
IPO        -0.0773       -0.3892      0.1986       -0.3161        -0.0150   
OLS        -0.0693       -0.8748      0.0792       -0.2242        -0.0082   

     MVO cost  
IPO    0.0042  
OLS    0.0009  
